In [0]:
import requests

from pyspark.sql.functions import current_timestamp

BASE_URL = "https://api.gbif.org/v1/occurrence/search"

LIMIT = 300
OFFSET = 0

dados_total = []

while True:

    params = {
        "country": "BR",
        "hasCoordinate": "true",
        "limit": LIMIT,
        "offset": OFFSET
    }

    response = requests.get(BASE_URL, params=params)

    payload = response.json()

    resultados = payload.get("results", [])

    if not resultados:
        break

    dados_total.extend(resultados)

    OFFSET += LIMIT

    if OFFSET >= 3000:
        break

dados_tratados = []

for r in dados_total:

    dados_tratados.append({
        "occurrence_id": str(r.get("key")),
        "species": str(r.get("species")) if r.get("species") else None,
        "genus": str(r.get("genus")) if r.get("genus") else None,
        "family": str(r.get("family")) if r.get("family") else None,
        "order_name": str(r.get("order")) if r.get("order") else None,
        "class_name": str(r.get("class")) if r.get("class") else None,
        "phylum": str(r.get("phylum")) if r.get("phylum") else None,
        "kingdom": str(r.get("kingdom")) if r.get("kingdom") else None,
        "latitude": float(r.get("decimalLatitude")) if r.get("decimalLatitude") else None,
        "longitude": float(r.get("decimalLongitude")) if r.get("decimalLongitude") else None,
        "event_date": str(r.get("eventDate")) if r.get("eventDate") else None,
        "country_code": str(r.get("countryCode")) if r.get("countryCode") else None
    })

df_bronze = (
    spark.createDataFrame(dados_tratados)
         .withColumn("dt_ingestao", current_timestamp())
)

display(df_bronze)

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable("gs_carbon.bronze.gbif_metadata")
)